In [ ]:
# Step: Mount local path to model
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#Step 3: Run model
MODEL_ID = "/content/drive/MyDrive/Colab Notebooks/Apollo2"
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
prompt = "Explain quantization in LLM"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        pad_token_id = eos_token_id,
        do_sample=True
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Explain quantization in LLM (Language Models) context

Quantization in the context of Language Models (LLMs) refers to the process of approximating the weights and biases of a language model using quantization techniques. Quantization reduces the precision of the weight values from their binary (0, 1) representation to a fixed-point representation by truncating the decimal part after a certain number of digits. 

The goal of quantization is to reduce the computational load and memory requirements of the language model while maintaining its performance. By approximating the weights and biases with a fixed-point representation, the number of computations needed to compute gradients and update the model parameters is reduced. 

In addition to reducing computational load, quantization can also help to improve the efficiency of inference by reducing the precision of the weight values. This can be particularly useful for deploying LLMs in resource-constrained environments, such as mobile dev

In [ ]:
#!git clone https://huggingface.co/onnx-community/Qwen3-0.6B-ONNX
#!git clone https://huggingface.co/FreedomIntelligence/Apollo2-1.5B

Cloning into 'Qwen3-0.6B-ONNX'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 39 (delta 8), reused 0 (delta 0), pack-reused 6 (from 1)
Receiving objects: 100% (39/39), 3.75 MiB | 5.13 MiB/s, done.
Resolving deltas: 100% (8/8), done.
Filtering content: 100% (9/9), 7.30 GiB | 15.42 MiB/s, done.


In [ ]:
#!git clone https://github.com/casper-hansen/AutoAWQ
#!cd AutoAWQ
#!pip install -e .
!pip install autoawq

In [ ]:
!curl https://raw.githubusercontent.com/microsoft/onnxruntime-genai/main/examples/python/awq-quantized-model.py -o awq-quantized-model.py

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  4548  100  4548    0     0  20679      0 --:--:-- --:--:-- --:--:-- 20767


In [ ]:
!python awq-quantized-model.py --model_path /Apollo2 --quant_path ./awq-out/ --output_path ./onnx-out/ --execution_provider cpu

In [ ]:
!pip install onnx_ir
!pip install onnxruntime-genai
!pip install transformers==4.51.3 -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 82.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.7.1
    Uninstalling huggingface_hub-1.7.1:
      Successfully uninstalled huggingface_hub-1.7.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0

In [ ]:
!cp -r "onnx-out" "/content/drive/MyDrive/Colab Notebooks/apollo2-onnx"

In [ ]:
import onnxruntime_genai as og
from onnxruntime_genai.models.builder import create_model
import json

# Load model
print("Loading model...")
config = og.Config('/content/drive/MyDrive/Colab Notebooks/apollo2-onnx/onnx-out/')
config.clear_providers()
model = og.Model(config)
print("Model loaded")
tokenizer = og.Tokenizer(model)

# Override any default search options in `genai_config.json`
search_options = {
        "diversity_penalty": 0.0,
        "do_sample": True,
        "early_stopping": True,
        "length_penalty": 1.0,
        "max_length": 200,
        "min_length": 1,
        "no_repeat_ngram_size": 0,
        "num_beams": 1,
        "num_return_sequences": 1,
        "past_present_share_buffer": True,
        "repetition_penalty": 1.1,
        "temperature": 0.7,
        "top_k": 40,
        "top_p": 0.9
}




Loading model...
Model loaded


In [ ]:
params = og.GeneratorParams(model)

params.set_search_options(
    max_length=128,
    temperature=0.0
)

generator = og.Generator(model, params)

prompt = "What is diabetes?"

input_tokens = tokenizer.encode(prompt)

in_len = len(input_tokens)

generator.append_tokens(input_tokens)

while not generator.is_done():
    generator.generate_next_token()

output_tokens = generator.get_sequence(0)

text = tokenizer.decode(output_tokens[in_len:])

print(text)

 Diabetes is a chronic disease that affects the body's ability to regulate blood sugar levels. It is caused by a lack of insulin, a hormone that helps cells absorb glucose from the bloodstream. Without insulin, the body cannot properly use and store the sugar it absorbs, leading to high blood sugar levels. This condition is known as hyperglycemia. The body tries to compensate for the lack of insulin by producing more insulin, but this can lead to the pancreas eventually running out of insulin. At this point, the body is unable to regulate blood sugar levels, and the condition becomes severe. In the most severe form,


In [ ]:
import numpy as np
logits = generator.get_logits()

logits = np.array(logits)

print(np.sum(logits,axis=2))

[[-317096.7]]


In [ ]:
import time
import json
import numpy as np

import onnxruntime_genai as og

from datasets import load_dataset
from tqdm import tqdm

# =========================
# CONFIG
# =========================

MODEL_PATH = '/content/drive/MyDrive/Colab Notebooks/apollo2-onnx/onnx-out/'

EVAL_SAMPLES = 500
MAX_LENGTH = 512

OUTPUT_FILE = "onnx_benchmark_results.json"

# =========================
# LOAD MODEL
# =========================

print("Loading ONNX model...")

model = og.Model(MODEL_PATH)

tokenizer = og.Tokenizer(model)

# =========================
# PROMPT FORMAT
# =========================

def format_medmcqa(sample):

    options = [
        sample["opa"],
        sample["opb"],
        sample["opc"],
        sample["opd"]
    ]

    prompt = f"""
Question:
{sample['question']}

Options:
A. {options[0]}
B. {options[1]}
C. {options[2]}
D. {options[3]}

Answer:
"""

    return prompt


def extract_letter(text):

    for letter in ["A", "B", "C", "D"]:
        if letter in text:
            return letter

    return "A"

# =========================
# GENERATION FUNCTION
# =========================

def generate_answer(prompt):

    tokens = tokenizer.encode(prompt)
    input_length = len(tokens)

    # Create fresh params per request
    params = og.GeneratorParams(model)

    params.set_search_options(
        max_length=input_length + MAX_LENGTH,
        temperature=0.0,
        do_sample=False
    )

    generator = og.Generator(model, params)


    start = time.time()
    generator.append_tokens(tokens)

    while not generator.is_done():
            generator.generate_next_token()

    end = time.time()

    output_tokens = generator.get_sequence(0)

    text = tokenizer.decode(output_tokens)

    answer = text[len(prompt):].strip()

    latency = end - start

    token_count = len(output_tokens)

    return answer, latency, token_count


# =========================
# MEDMCQA BENCHMARK
# =========================

print("\nRunning MedMCQA...")

dataset = load_dataset(
    "medmcqa",
    split="validation"
)

correct = 0

latencies = []
tokens_total = 0

for sample in tqdm(dataset.select(range(EVAL_SAMPLES))):

    prompt = format_medmcqa(sample)

    answer_text, latency, tokens = generate_answer(prompt)

    pred_letter = extract_letter(answer_text)

    correct_idx = sample["cop"]

    correct_letter = ["A","B","C","D"][correct_idx]

    if pred_letter == correct_letter:
        correct += 1

    latencies.append(latency)
    tokens_total += tokens


medmcqa_accuracy = correct / EVAL_SAMPLES

# =========================
# PUBMEDQA BENCHMARK
# =========================

print("\nRunning PubMedQA...")

pubmedqa = load_dataset(
    "pubmed_qa",
    "pqa_labeled",
    split="train"
)

correct = 0

for sample in tqdm(pubmedqa.select(range(EVAL_SAMPLES))):

    context = " ".join(
        sample["context"]["contexts"]
    )

    question = sample["question"]

    prompt = f"""
Context:
{context}

Question:
{question}

Answer yes, no, or maybe.

Answer:
"""

    answer_text, latency, tokens = generate_answer(prompt)

    prediction = answer_text.lower()

    gold = sample["final_decision"].lower()

    if gold in prediction:
        correct += 1


pubmedqa_accuracy = correct / EVAL_SAMPLES

# =========================
# PERFORMANCE METRICS
# =========================

avg_latency = np.mean(latencies)

tokens_per_sec = tokens_total / np.sum(latencies)

# =========================
# SAVE RESULTS
# =========================

results = {

    "model_type": "onnxruntime-genai",

    "model_path": MODEL_PATH,

    "medmcqa_accuracy": medmcqa_accuracy,

    "pubmedqa_accuracy": pubmedqa_accuracy,

    "avg_latency_sec": float(avg_latency),

    "tokens_per_sec": float(tokens_per_sec)
}

with open(OUTPUT_FILE, "w") as f:

    json.dump(results, f, indent=4)

print("\nFinal Results:")

print(json.dumps(results, indent=4))

Loading ONNX model...

Running MedMCQA...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/85.9M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/936k [00:00<?, ?B/s]

...

README.md: 0.00B [00:00, ?B/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

100%|██████████| 500/500 [2:31:17<00:00, 18.16s/it]


Final Results:
{
    "model_type": "onnxruntime-genai",
    "model_path": "/content/drive/MyDrive/Colab Notebooks/apollo2-onnx/onnx-out/",
    "medmcqa_accuracy": 0.37,
    "pubmedqa_accuracy": 0.692,
    "avg_latency_sec": 5.4089122796058655,
    "tokens_per_sec": 16.38407047829911
}
